<a href="https://colab.research.google.com/github/KishorS-eu/shotdistributionpredictor/blob/main/ShotEventProcessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas numpy statsbombpy
import pandas as pd
import numpy as np
from statsbombpy import sb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 6.1 MB/s eta 0:00:00


In [ ]:
def get_teamsheet(lineup):
    player_data = set()
    for player in lineup['lineup']:
        player_data.add((str(player['player']['id']), player['player']['name']))
    return frozenset(player_data)

In [ ]:
def sub_teamsheet(lineup, subin_id, subin_name, subout_id, subout_name):
    mut_lineup = set(lineup)
    mut_lineup.remove((str(int(subout_id)), subout_name))
    mut_lineup.add((str(int(subin_id)), subin_name))
    return frozenset(mut_lineup)

In [ ]:
def playeroff_teamsheet(lineup, out_id, out_name):
    mut_lineup = set(lineup)
    mut_lineup.remove((str(int(out_id)), out_name))
    return frozenset(mut_lineup)

In [ ]:
def playeron_teamsheet(lineup, on_id, on_name):
    mut_lineup = set(lineup)
    mut_lineup.add((str(int(on_id)), on_name))
    return frozenset(mut_lineup)

In [ ]:
def mandown_teamsheet(lineup):
    return len(lineup) < 11

In [ ]:
def get_lineup_events(team_id, match_id):
    selected_types = ['Starting XI', 'Half End', 'Substitution',
                     'Player On', 'Player Off']

    # get match event data
    event_df = sb.events(match_id = match_id)
    filtered_df = event_df[event_df['type'].isin(selected_types)]
    filtered_df = filtered_df[filtered_df['team_id'] == team_id]
    filtered_df = filtered_df.sort_values(by = ['period', 'timestamp']).copy()
    filtered_df = filtered_df[ ~((filtered_df['type'] == 'Half End') &
                                (filtered_df['period'] == 1))]
    filtered_df = filtered_df.reset_index(drop = True)


    # get the starting lineup array
    starting_xi_event = filtered_df[
        (filtered_df['type'] == 'Starting XI') &
        (filtered_df['team_id'] == team_id)
        ].iloc[0]

    starting_xi = starting_xi_event['tactics']
    starting_ts = get_teamsheet(starting_xi)

    teamsheets = [starting_ts]
    mandown = []
    for idx, row in filtered_df.iterrows():
        event_type = row['type']
        # handling changes in teamsheets for different event types
        if event_type == 'Starting XI':
            pass

        elif event_type == 'Substitution':
            subout_id = row['player_id']
            subout_name = row['player']
            subin_id = row['substitution_replacement_id']
            subin_name = row['substitution_replacement']
            teamsheets.append(sub_teamsheet(teamsheets[idx - 1], subin_id,
                                            subin_name, subout_id, subout_name))

        elif event_type == 'Player Off':
            out_id = row['player_id']
            out_name = row['player']
            teamsheets.append(playeroff_teamsheet(teamsheets[idx - 1], out_id,
                                                  out_name))

        elif event_type == 'Player On':
            on_id = row['player_id']
            on_name = row['player']
            teamsheets.append(playeron_teamsheet(teamsheets[idx - 1], on_id,
                                                 on_name))

        elif event_type == 'Half End':
            teamsheets.append(teamsheets[idx - 1])

        # adding boolean value to mandown based on if the teamsheet has a numpy nan value
        mandown.append(mandown_teamsheet(teamsheets[idx]))

    filtered_df['teamsheet'] = teamsheets
    filtered_df['mandown'] = mandown
    filtered_df.loc[:, 'teamsheet'] = filtered_df['teamsheet']

    columns_to_retain = ['id', 'index', 'match_id', 'team', 'team_id', 'period',
                         'timestamp', 'type', 'teamsheet', 'mandown']

    return filtered_df[columns_to_retain].reset_index(drop=True)

In [ ]:
def get_shots_from_timeline(team_id, match_id, lineup_events):
    events = sb.events(match_id = match_id)
    shots_df = events[(events['type'] == 'Shot') & (events['team_id'] == team_id)].copy()
    shots_df = shots_df.sort_values(by=['period', 'timestamp']).reset_index(drop=True)

    shot_teamsheets = []
    shot_mandown = []

    for idx, shot in shots_df.iterrows():
        shot_period = shot['period']
        shot_time = shot['timestamp']

        past_events = lineup_events[
            (lineup_events['period'] == shot_period) &
            (lineup_events['timestamp'] <= shot_time)
        ]

        if len(past_events) > 0:
            active_state = past_events.iloc[-1]
        else:
            active_state = lineup_events[(lineup_events['period'] == shot_period - 1)].iloc[-1]

        shot_teamsheets.append(active_state['teamsheet'])
        shot_mandown.append(active_state['mandown'])

    shots_df['teamsheet'] = shot_teamsheets
    shots_df['mandown'] = shot_mandown

    return shots_df.dropna(axis = 1, how = 'all')

In [ ]:
def get_team_matchids(comp_id, season_id, team_id):
    season_matches = sb.matches(competition_id = comp_id, season_id = season_id)
    team_matches = season_matches[(season_matches['home_team_id'] == team_id)
                                  |(season_matches['away_team_id'] == team_id)].reset_index(drop=True)
    return team_matches['match_id'].to_list()

In [ ]:
def get_teamseason_shot_events(comp_id, season_id, team_id):
    team_matchids = get_team_matchids(comp_id, season_id, team_id)

    games = len(team_matchids)
    print(f'Found {games} games for the season')

    team_shot_events = get_shots_from_timeline(team_id, team_matchids[0], get_lineup_events(team_id, team_matchids[0]))
    print(f'Processed shot event data for game 1/{games}')

    for id_x, id_game in enumerate(team_matchids[1:]):
        team_shot_events = pd.concat([team_shot_events, get_shots_from_timeline(team_id, id_game, get_lineup_events(team_id, id_game))])
        print(f'Processed shot event data for game {id_x + 2}/{games}')
    return team_shot_events

In [ ]:
def get_uniquelineups(shotevents_df):
    unique_lineups = shotevents_df['teamsheet'].unique()

    lineup_df= []
    for idx, l_key in enumerate(unique_lineups):
        subset_df = shotevents_df[shotevents_df['teamsheet'] == l_key].copy()
        lineup_df.append([subset_df, l_key])

    return lineup_df

In [ ]:
def get_allfeaturedplayers(shotevents_df):
    unique_lineups = shotevents_df['teamsheet'].unique()

    listoflineups = list(unique_lineups)
    return frozenset().union(*listoflineups)

In [ ]:
def get_playerusage(processedshots_df, player_id, player_name):

    sublist = [
        (df, f_set) for df, f_set in processedshots_df
        if (str(int(player_id)), player_name) in f_set
    ]
    subarray = np.array(sublist, dtype = object)
    new_df = pd.concat(list(subarray[:, 0]))
    new_df = new_df[(~new_df['mandown'])]
    player_df = new_df[(new_df['player_id'] == player_id)]

    return [len(player_df), len(new_df), (100*(len(player_df)/len(new_df))), new_df, player_df]

In [ ]:
def get_teams(comp_id, season_id):
    season_matches = sb.matches(competition_id = comp_id, season_id = season_id)
    return season_matches[['home_team_id', 'home_team']].copy().drop_duplicates().sort_values(by = ['home_team_id']).reset_index(drop=True)